In [ ]:
import pandas as pd
import numpy as np
import pickle
import shap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

DATA    = Path.cwd().parent / 'data' / 'interim'
FIGURES = Path.cwd().parent / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DAY      = 30
ONBOARDING_WINDOW = 90

eligible          = pd.read_csv(DATA / 'eligible_with_labels.csv')
new_clients       = pd.read_csv(DATA / 'new_clients.csv')
tx_with_reg       = pd.read_csv(DATA / 'tx_with_reg.csv')
messages          = pd.read_csv(DATA / 'messages.csv')
message_templates = pd.read_csv(DATA / 'message_templates.csv')
conversions       = pd.read_csv(DATA / 'conversions.csv')
business_units    = pd.read_csv(DATA / 'business_units.csv')

for df, col in [
    (new_clients, 'registration_date'),
    (tx_with_reg, 'date'),
    (tx_with_reg, 'registration_date'),
    (messages,    'send_date'),
    (conversions, 'date'),
]:
    df[col] = pd.to_datetime(df[col], utc=True)

# Compute days_since_first_tx: days relative to each client's first transaction
first_tx_reg_day = tx_with_reg.groupby('client_id')['days_since_reg'].min().rename('first_tx_reg_day')
tx_with_reg = tx_with_reg.join(first_tx_reg_day, on='client_id')
tx_with_reg['days_since_first_tx'] = tx_with_reg['days_since_reg'] - tx_with_reg['first_tx_reg_day']

print(f'eligible: {len(eligible):,}')


In [ ]:
# Multi-visit subset: clients with > 1 transaction in first 30 days
tx_snapshot_all = tx_with_reg[
    tx_with_reg['client_id'].isin(eligible['client_id']) &
    (tx_with_reg['days_since_first_tx'] >= 0) &
    (tx_with_reg['days_since_first_tx'] <= SNAPSHOT_DAY)
].copy()
multi_visit_ids = tx_snapshot_all.groupby('client_id')['transaction_id'].count()
multi_visit_ids = multi_visit_ids[multi_visit_ids > 1].index

eligible_mv = eligible[eligible['client_id'].isin(multi_visit_ids)].copy()
tx_snapshot  = tx_snapshot_all[tx_snapshot_all['client_id'].isin(multi_visit_ids)].copy()
tx_snapshot  = tx_snapshot.merge(
    business_units[['id','location','category']], left_on='business_unit_id', right_on='id', how='left'
)

first_tx = (
    tx_with_reg.groupby('client_id')['days_since_reg'].min()
    .reset_index().rename(columns={'days_since_reg':'day_first_tx'})
)

format_median_gap = (
    tx_with_reg[tx_with_reg['client_id'].isin(eligible['client_id'])]
    .merge(business_units[['id','location']], left_on='business_unit_id', right_on='id', how='left')
    .assign(tx_date=lambda x: x['days_since_first_tx'].round(0))
    .drop_duplicates(subset=['client_id','tx_date'])
    .sort_values(['client_id','tx_date'])
    .assign(prev_day=lambda x: x.groupby('client_id')['tx_date'].shift(1))
    .assign(gap=lambda x: x['tx_date'] - x['prev_day'])
    .dropna(subset=['gap'])
    .groupby('location')['gap'].median()
    .reset_index().rename(columns={'gap':'median_gap','location':'first_location'})
)
global_median = format_median_gap['median_gap'].median()

print(f'All eligible:         {len(eligible):,}')
print(f'Multi-visit (30d):    {len(eligible_mv):,}  ({len(eligible_mv)/len(eligible)*100:.1f}%)')
print(f'Churn rate (all):     {eligible["churned"].mean()*100:.1f}%')
print(f'Churn rate (mv only): {eligible_mv["churned"].mean()*100:.1f}%')


In [ ]:
# RFM + format-normalised features (same as NB08 but on mv subset)
tx_count = (tx_snapshot.groupby('client_id')['transaction_id'].count()
            .reset_index().rename(columns={'transaction_id':'tx_count'}))
unique_days = (
    tx_snapshot.assign(tx_date=lambda x: x['days_since_first_tx'].round(0))
    .drop_duplicates(subset=['client_id','tx_date'])
    .groupby('client_id')['tx_date'].count()
    .reset_index().rename(columns={'tx_date':'unique_active_days'})
)
recency = (tx_snapshot.groupby('client_id')['days_since_first_tx'].max()
           .reset_index().rename(columns={'days_since_first_tx':'last_tx_day'}))
recency['recency'] = SNAPSHOT_DAY - recency['last_tx_day']
recency = recency[['client_id','recency']]
monetary = (tx_snapshot.groupby('client_id').agg(
    total_spent=('payed_amount','sum'), avg_check=('payed_amount','mean'),
    bonuses_accum=('bonusses_accum','sum'), bonuses_used=('bonusses_used','sum')
).reset_index())
tx_sorted = (
    tx_snapshot.assign(tx_date=lambda x: x['days_since_first_tx'].round(0))
    .drop_duplicates(subset=['client_id','tx_date']).sort_values(['client_id','tx_date'])
)
tx_sorted['prev_day'] = tx_sorted.groupby('client_id')['tx_date'].shift(1)
tx_sorted['gap']      = tx_sorted['tx_date'] - tx_sorted['prev_day']
avg_gap = (tx_sorted.groupby('client_id')['gap'].mean()
           .reset_index().rename(columns={'gap':'avg_days_between_tx'}))
avg_gap['avg_days_between_tx'] = avg_gap['avg_days_between_tx'].fillna(0)

# gap_trend: positive = widening gaps (risky signal)
gap_first = tx_sorted.groupby('client_id')['gap'].first().reset_index().rename(columns={'gap':'gap_first'})
gap_last  = tx_sorted.groupby('client_id')['gap'].last().reset_index().rename(columns={'gap':'gap_last'})
gap_trend = gap_first.merge(gap_last, on='client_id', how='left')
gap_trend['gap_trend'] = gap_trend['gap_last'] - gap_trend['gap_first']
gap_trend = gap_trend[['client_id','gap_trend']].fillna(0)

half = SNAPSHOT_DAY // 2
first_half  = (tx_snapshot[tx_snapshot['days_since_first_tx'] <= half].groupby('client_id')['transaction_id'].count()
               .reset_index().rename(columns={'transaction_id':'tx_first_half'}))
second_half = (tx_snapshot[tx_snapshot['days_since_first_tx'] > half].groupby('client_id')['transaction_id'].count()
               .reset_index().rename(columns={'transaction_id':'tx_second_half'}))
trend = first_half.merge(second_half, on='client_id', how='outer').fillna(0)
trend['activity_trend'] = trend['tx_second_half'] - trend['tx_first_half']
trend = trend[['client_id','activity_trend']]

first_location = (
    tx_snapshot.sort_values('days_since_first_tx')
    .groupby('client_id')[['location','category']].first().reset_index()
    .rename(columns={'location':'first_location','category':'first_category'})
)
first_location = first_location.merge(format_median_gap, on='first_location', how='left')
first_location['median_gap'] = first_location['median_gap'].fillna(global_median)
unique_locations = (tx_snapshot.groupby('client_id')['location'].nunique()
                    .reset_index().rename(columns={'location':'unique_locations'}))
online_share = (tx_snapshot.groupby('client_id')
                .apply(lambda x: (x['category'] == 'Online purchases').mean())
                .reset_index().rename(columns={0:'online_tx_share'}))
recency_norm = recency.merge(first_location[['client_id','median_gap']], on='client_id', how='left')
recency_norm['recency_normalized'] = recency_norm['recency'] / recency_norm['median_gap']
recency_norm = recency_norm[['client_id','recency_normalized']]
visit_fulfill = unique_days.merge(first_location[['client_id','median_gap']], on='client_id', how='left')
visit_fulfill['expected_visits'] = SNAPSHOT_DAY / visit_fulfill['median_gap']
visit_fulfill['visit_fulfillment'] = visit_fulfill['unique_active_days'] / visit_fulfill['expected_visits']
visit_fulfill = visit_fulfill[['client_id','visit_fulfillment']]
print('RFM + gap_trend + format-normalised features done')


In [ ]:
# Communication and conversion features (mv clients only)
segment_groups = {
    'Thank-you / post-visit':'auto_trigger','Quest / cross-venue':'auto_trigger',
    'Loyalty: double points':'retention_offer','Loyalty: points spend/expiry':'retention_offer',
    'Coupon / discount':'retention_offer',
}
message_templates['segment_group'] = message_templates['segment'].map(segment_groups).fillna('other')
bridge = new_clients[['client_id','loyalty_client_id']].copy()
messages_new = (
    messages.dropna(subset=['client_id'])
    .merge(bridge, left_on='client_id', right_on='loyalty_client_id', how='inner')
    .rename(columns={'client_id_x':'loyalty_client_id_msg','client_id_y':'client_id'})
    .merge(new_clients[['client_id','registration_date']], on='client_id', how='left')
    .merge(first_tx, on='client_id', how='left')
)
messages_new['days_since_first_tx'] = (
    (messages_new['send_date'] - messages_new['registration_date']).dt.total_seconds() / 86400
    - messages_new['day_first_tx']
)
msg_snapshot = messages_new[
    (messages_new['days_since_first_tx'] >= 0) &
    (messages_new['days_since_first_tx'] <= SNAPSHOT_DAY) &
    (messages_new['client_id'].isin(eligible_mv['client_id']))
].merge(message_templates[['message_template_id','segment_group']], on='message_template_id', how='left')
msg_features = (
    msg_snapshot.groupby('client_id')
    .agg(_n=('message_id','count'), _op=('status', lambda x: (x=='Opened').sum()))
    .reset_index()
)
msg_features['open_rate'] = msg_features['_op'] / msg_features['_n']
msg_features = msg_features[['client_id','open_rate']]
conv_new = (
    conversions
    .merge(bridge, left_on='client_id', right_on='loyalty_client_id', how='inner')
    .rename(columns={'client_id_x':'loyalty_client_id_conv','client_id_y':'client_id'})
    .merge(new_clients[['client_id','registration_date']], on='client_id', how='left')
    .merge(first_tx, on='client_id', how='left')
)
conv_new['days_since_first_tx'] = (
    (conv_new['date'] - conv_new['registration_date']).dt.total_seconds() / 86400
    - conv_new['day_first_tx']
)
conv_snapshot = conv_new[
    (conv_new['days_since_first_tx'] >= 0) &
    (conv_new['days_since_first_tx'] <= SNAPSHOT_DAY) &
    (conv_new['client_id'].isin(eligible_mv['client_id']))
]
conv_features = (
    conv_snapshot.groupby('client_id').agg(conv_count=('conversion_id','count')).reset_index()
)
profile = new_clients[new_clients['client_id'].isin(eligible_mv['client_id'])][['client_id','registration_date']].copy()
profile['registration_date'] = pd.to_datetime(profile['registration_date']).dt.tz_localize(None)
profile['reg_day_of_week'] = profile['registration_date'].dt.dayofweek
profile['reg_month']       = profile['registration_date'].dt.month
profile = profile[['client_id','reg_day_of_week','reg_month']]
print('Communication and conversion features done')


In [ ]:
# Assemble mv feature matrix
features_mv = eligible_mv[['client_id','churned']].copy()
for df in [tx_count, unique_days, recency, monetary, avg_gap, gap_trend, trend,
           first_location[['client_id','first_location','first_category']],
           unique_locations, online_share, recency_norm, visit_fulfill]:
    features_mv = features_mv.merge(df, on='client_id', how='left')
features_mv = features_mv.merge(msg_features, on='client_id', how='left')
features_mv = features_mv.merge(conv_features, on='client_id', how='left')
features_mv = features_mv.merge(profile, on='client_id', how='left')
for col in ['avg_check','total_spent','activity_trend','gap_trend','open_rate',
            'conv_count','bonuses_accum','bonuses_used','unique_locations','online_tx_share']:
    features_mv[col] = features_mv[col].fillna(0)
print(f'features_mv: {features_mv.shape}  churn: {features_mv["churned"].mean()*100:.1f}%')
print(f'NaN remaining: {features_mv.isna().sum().sum()}')


In [ ]:
# Temporal split + train LR, RF, XGBoost on mv features
features_mv = features_mv.merge(new_clients[['client_id','registration_date']], on='client_id', how='left')
features_mv['registration_date'] = pd.to_datetime(features_mv['registration_date'], utc=True)
train_mask = features_mv['registration_date'] < pd.Timestamp('2031-06-01', tz='UTC')
val_mask   = (features_mv['registration_date'] >= pd.Timestamp('2031-06-01', tz='UTC')) & \
             (features_mv['registration_date'] <  pd.Timestamp('2031-08-01', tz='UTC'))
test_mask  = features_mv['registration_date'] >= pd.Timestamp('2031-08-01', tz='UTC')
print(f'Train: {train_mask.sum():,}  Val: {val_mask.sum():,}  Test: {test_mask.sum():,}')
for name, mask in [('Train',train_mask),('Val',val_mask),('Test',test_mask)]:
    print(f'  Churn {name}: {features_mv.loc[mask,"churned"].mean()*100:.1f}%')

DROP_COLS = ['client_id','churned','registration_date']
CAT_COLS  = ['first_location','first_category']
df_train = features_mv[train_mask].copy()
df_val   = features_mv[val_mask].copy()
df_test  = features_mv[test_mask].copy()
le_dict = {}
for col in CAT_COLS:
    le = LabelEncoder(); le.fit(df_train[col])
    df_train[col] = le.transform(df_train[col])
    df_val[col]   = df_val[col].map(dict(zip(le.classes_, le.transform(le.classes_)))).fillna(-1).astype(int)
    df_test[col]  = df_test[col].map(dict(zip(le.classes_, le.transform(le.classes_)))).fillna(-1).astype(int)
    le_dict[col]  = le
X_train = df_train.drop(columns=DROP_COLS); y_train = df_train['churned']
X_val   = df_val.drop(columns=DROP_COLS);   y_val   = df_val['churned']
X_test  = df_test.drop(columns=DROP_COLS);  y_test  = df_test['churned']

lr_mv = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_mv.fit(X_train, y_train)
rf_mv = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf_mv.fit(X_train, y_train)
rf_test_proba = rf_mv.predict_proba(X_test)[:,1]
rf_test_auc   = roc_auc_score(y_test, rf_test_proba)
scale_pw = (y_train == 0).sum() / (y_train == 1).sum()
xgb_mv = xgb.XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pw,
    random_state=42, eval_metric='auc', early_stopping_rounds=50, verbosity=0)
xgb_mv.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_test_auc = roc_auc_score(y_test, xgb_mv.predict_proba(X_test)[:,1])

print(f"{'Model':<22} {'Val AUC':>9} {'Test AUC':>9}")
print('-' * 44)
print(f"  {'Logistic Regression':<20} {roc_auc_score(y_val, lr_mv.predict_proba(X_val)[:,1]):>9.4f} {roc_auc_score(y_test, lr_mv.predict_proba(X_test)[:,1]):>9.4f}")
print(f"  {'Random Forest':<20} {roc_auc_score(y_val, rf_mv.predict_proba(X_val)[:,1]):>9.4f} {rf_test_auc:>9.4f}")
print(f"  {'XGBoost':<20} {roc_auc_score(y_val, xgb_mv.predict_proba(X_val)[:,1]):>9.4f} {xgb_test_auc:>9.4f}")


In [ ]:
# Save prod model and predictions
with open(DATA / 'best_model_rf_mv.pkl', 'wb') as f:
    pickle.dump(rf_mv, f)
X_test.to_csv(DATA / 'X_test_mv.csv', index=False)
y_test.to_csv(DATA / 'y_test_mv.csv', index=False)
pd.DataFrame({
    'client_id':    df_test['client_id'].values,
    'churned':      y_test.values,
    'proba_rf_mv':  rf_test_proba,
    'proba_xgb_mv': xgb_mv.predict_proba(X_test)[:,1],
    'proba_lr_mv':  lr_mv.predict_proba(X_test)[:,1],
}).to_csv(DATA / 'test_predictions_mv.csv', index=False)
print('Saved: best_model_rf_mv.pkl, X/y_test_mv.csv, test_predictions_mv.csv')


In [ ]:
# SHAP for prod model
print('Computing SHAP values for prod model...')
explainer_mv = shap.TreeExplainer(rf_mv)
shap_values_mv = explainer_mv.shap_values(X_test)
shap_mv = shap_values_mv[:, :, 1]
np.save(DATA / 'shap_churned_mv.npy', shap_mv)

mean_shap = pd.Series(np.abs(shap_mv).mean(axis=0), index=X_test.columns)
FORMAT_NORM = {'recency_normalized', 'visit_fulfillment'}
print(f"{'Rank':>4} {'Feature':<30} {'Mean|SHAP|':>11}")
print('-' * 48)
for i, (feat, val) in enumerate(mean_shap.sort_values(ascending=False).items(), 1):
    tag = '  <-- format-norm' if feat in FORMAT_NORM else ''
    print(f'{i:>4}  {feat:<30} {val:>11.4f}{tag}')


In [ ]:
# SHAP figures (overwrite NB05 figures with prod model versions)
plt.figure(figsize=(9, 7))
colors_bar = ['#e07b39' if f in {'recency_normalized','visit_fulfillment'} else '#2c6fad'
              for f in mean_shap.sort_values().index]
plt.barh(mean_shap.sort_values().index, mean_shap.sort_values().values,
         color=colors_bar, edgecolor='white', height=0.7)
plt.xlabel('Mean |SHAP value|', fontsize=11)
plt.title('SHAP -- Global Feature Importance (Prod 30d mv RF)', fontsize=12, fontweight='bold')
plt.legend(handles=[
    mpatches.Patch(color='#e07b39', label='Format-normalised'),
    mpatches.Patch(color='#2c6fad', label='Other'),
], fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES / 'shap_global_bar.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(9, 8))
shap.summary_plot(shap_mv, X_test, max_display=15, show=False)
plt.title('SHAP Beeswarm -- Prod 30d mv RF', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(FIGURES / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Per-format AUC breakdown
results_mv = pd.read_csv(DATA / 'test_predictions_mv.csv')
features_all = pd.read_csv(DATA / 'features.csv')
test_df = results_mv.merge(features_all[['client_id','first_location']], on='client_id', how='left')

FORMAT_EN = {
    'Restaurant':'Restaurant', "Kav'yarniya":'Coffee shop',
    'Bar':'Bar', 'Souvenir shop':'Souvenir shop',
    'Cafe':'Cafe', 'Pub':'Pub', 'Delivery':'Delivery',
}
auc_overall = roc_auc_score(test_df['churned'], test_df['proba_rf_mv'])

print(f"{'Format':<18} {'N':>7} {'Churn%':>8} {'AUC':>8}")
print('-' * 44)
for fmt in test_df['first_location'].unique():
    sub = test_df[test_df['first_location'] == fmt]
    if len(sub) < 30 or sub['churned'].nunique() < 2: continue
    auc   = roc_auc_score(sub['churned'], sub['proba_rf_mv'])
    print(f'  {fmt:<16} {len(sub):>7,} {sub["churned"].mean()*100:>7.1f}%  {auc:>7.4f}')
print(f'  {"Overall":<16} {len(test_df):>7,} {test_df["churned"].mean()*100:>7.1f}%  {auc_overall:>7.4f}  <--')
